# 1. Library import

In [ ]:
! pip install spacy fr_core_news_md

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 17.7 MB/s eta 0:00:00


In [ ]:
# General use
import pandas as pd
import numpy as np

In [ ]:
# Data preparation
from collections import Counter
from collections import defaultdict
import nltk
import string
from math import nan
import re
from typing import List, Tuple, Dict, Any

In [ ]:
# Tokenization
import spacy
nlp = spacy.load("fr_core_news_md")

# 2. Data import

In [ ]:
df = pd.read_csv('data_pub.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8938 entries, 0 to 8937
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         8938 non-null   int64 
 1   Fiche              8938 non-null   int64 
 2   year               8938 non-null   int64 
 3   Marque             8938 non-null   object
 4   Produit            8938 non-null   object
 5   Date               8938 non-null   object
 6   Support            8938 non-null   object
 7   Format             8938 non-null   object
 8   Secteur            8938 non-null   object
 9   Classe             8938 non-null   object
 10  Groupe             8938 non-null   object
 11  Variété            8938 non-null   object
 12  N° Groupe/Variété  8938 non-null   int64 
 13  Signature          4589 non-null   object
 14  Script             8515 non-null   object
 15  Incrustation       8930 non-null   object
 16  Titre              8938 non-null   object


In [ ]:
df["Script"].head(6)

,Script
0,"L'homme (1) : "" Wou !\r\n (1) ... On va où Mon..."
1,"Voix homme : "" 85 000 conducteurs vivent la se..."
2,"La femme (1) : "" C'est votre voiture ?\r\n L'h..."
3,"La femme (1) : "" C'est votre voiture ?\r\n L'h..."
4,"La femme (1) : "" C'est votre voiture ?\r\n L'h..."
5,"Voix homme : "" Pas de montant central, pas d'o..."


In [ ]:
print(df["Script"][1])

Voix homme : " 85 000 conducteurs vivent la sensation ...
 ... 100% électrique au volant de la nouvelle NISSAN LEAF.
 
 ... Rejognez un nouveau courant, et dites adieu à l'essence.
 
 ... Nouvelle NISSAN LEAF 100% électrique à partir de 169€ par mois. "


# 3. Maniuplation functions

In [ ]:
import re

def process_single_script(script_content: str) -> List[str]:
    """
    Processes a single script string, separating it into individual sentences.

    Args:
        script_content (str): The script text to process.

    Returns:
        List[str]: A list of processed sentences from the script.
    """
    processed_sentences = []

    if pd.isna(script_content):
        return [] # Return an empty list for missing scripts

    # Split into lines first, as per user request to treat \r\n as a primary separator
    lines = script_content.split('\r\n')
    if len(lines) == 1 and '\n' in script_content: # If no \r\n, try regular \n
        lines = script_content.split('\n')

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Step 1: Replace "..." with a placeholder so SpaCy doesn't incorrectly split sentences on these dots.
        # Use a more unique placeholder to avoid accidental matches and ensure clean reversion.
        placeholder = "___COLAB_ELLIPSIS_PLACEHOLDER___"
        processed_line = line.replace("...", placeholder)

        # Apply SpaCy's sentence segmentation to each processed line.
        doc = nlp(processed_line)
        for sent in doc.sents:
            sentence_text = sent.text

            # Step 2: Revert the unique placeholder back to "..."
            sentence_text = sentence_text.replace(placeholder, "...")

            # Step 3: Explicitly remove any remaining *fragments* of the placeholder if SpaCy broke it up.
            # This addresses the user's request to "Supprime les placeholders `___ELLIPSIS___` et `___`"
            # if they appear as artifacts, while preserving legitimate "..."
            sentence_text = sentence_text.replace("___", "") # Remove generic '___' fragments
            sentence_text = sentence_text.replace("COLAB", "") # Remove 'COLAB' if it's left as a fragment
            sentence_text = sentence_text.replace("ELLIPSIS", "") # Remove 'ELLIPSIS' if it's left as a fragment
            sentence_text = sentence_text.replace("PLACEHOLDER", "") # Remove 'PLACEHOLDER' if it's left as a fragment

            # Apply user's requested cleaning logic
            if not sentence_text:
                continue # Skip if empty after initial processing

            sentence_text = sentence_text.strip()

            # Remove surrounding quotes if present
            if sentence_text.startswith('"') and sentence_text.endswith('"'):
                sentence_text = sentence_text[1:-1]
            # Also remove all internal quotes
            sentence_text = sentence_text.replace('"', '')

            # Final strip after all modifications
            sentence_text = sentence_text.strip()

            # --- Refined filtering for artifacts ---

            # Filter out speaker tags if they appear as standalone sentences (e.g., 'Voix homme :', 'L'homme (1) :')
            # Fixed SyntaxError: unterminated string literal by putting regex on a single line.
            if re.fullmatch(r"^(L\'homme|L\'femme)\s*(\(\d+\))?\s*:?|Voix\s*(homme|femme)\s*:?$", sentence_text, re.IGNORECASE):
                continue

            # Filter out numerical or ellipsis-only fragments (e.g., '1)', '1) ...', '...', '.', '(', ')')
            # This regex now explicitly catches standalone '(', ')', numerical fragments and pure ellipsis/punctuation.
            if re.fullmatch(r'^\s*(\d+\))?\s*\.{0,3}\s*$|^[.,;:!?-]$', sentence_text):
                continue

            # Handle sentences starting with a comma (e.g., ', c'est...')
            if sentence_text.startswith(','):
                sentence_text = sentence_text.lstrip(',').strip()
                if not sentence_text: # If it becomes empty after stripping the comma
                    continue

            # Final check for empty string after all cleaning and filtering
            if sentence_text:
                processed_sentences.append(sentence_text)

    return processed_sentences

# 4. Apply data preparation

Applying `process_single_script` to the first 10 scripts and reconstructing `sentences_df` for testing.

In [ ]:
sentences_data_test = []

for original_idx, row in df.head(6).iterrows():
    script = row['Script']
    original_id = row['Fiche']

    processed_sents = process_single_script(script)

    if not processed_sents and pd.isna(script):
        sentences_data_test.append({
            'Fiche': original_id,
            'Sentence': nan,
            'Original_Index': original_idx
        })
    else:
        for sent in processed_sents:
            sentences_data_test.append({
                'Fiche': original_id,
                'Sentence': sent,
                'Original_Index': original_idx
            })

sentences_df_test = pd.DataFrame(sentences_data_test)

In [ ]:
display(sentences_df_test)

,Fiche,Sentence,Original_Index
0,4704876,L'homme (1) : Wou !,0
1,4704876,(,0
2,4704876,On va où Monsieur ?,0
3,4704876,L'homme (2) : Dans le 15ème !,0
4,4704876,(,0
...,...,...,...
105,4706394,C1,4
106,4706394,vitamine climatisé est à seulement 7 790 euros.,4
107,4707557,"Pas de montant central, pas d'obstacle.",5
108,4707557,"... FORD B-MAX, avec système de portes Easy Ac...",5


In [ ]:
for i in range(10):
  print(f'phrase {i}')
  print(sentences_df_test["Sentence"][i])

phrase 0
L'homme (1) :  Wou !
phrase 1
(
phrase 2
On va où Monsieur ?
phrase 3
L'homme (2) :  Dans le 15ème !
phrase 4
(
phrase 5
Ça m'arrange pas des masses !
phrase 6
Je veux bien aller à bon port, mais vous me dites bon port ...
phrase 7
La femme (3) :  Mais bon port c'est une expression !
phrase 8
(
phrase 9
J'ai vraiment mieux à vous proposer.


In [ ]:
display(sentences_df_test)
for i in range(len(sentences_df_test)):
  print(f'phrase {i}')
  print(sentences_df_test["Sentence"][i])

,Fiche,Sentence,Original_Index
0,4704876,L'homme (1) : Wou !,0
1,4704876,(,0
2,4704876,On va où Monsieur ?,0
3,4704876,L'homme (2) : Dans le 15ème !,0
4,4704876,(,0
...,...,...,...
105,4706394,C1,4
106,4706394,vitamine climatisé est à seulement 7 790 euros.,4
107,4707557,"Pas de montant central, pas d'obstacle.",5
108,4707557,"... FORD B-MAX, avec système de portes Easy Ac...",5


phrase 0
L'homme (1) :  Wou !
phrase 1
(
phrase 2
On va où Monsieur ?
phrase 3
L'homme (2) :  Dans le 15ème !
phrase 4
(
phrase 5
Ça m'arrange pas des masses !
phrase 6
Je veux bien aller à bon port, mais vous me dites bon port ...
phrase 7
La femme (3) :  Mais bon port c'est une expression !
phrase 8
(
phrase 9
J'ai vraiment mieux à vous proposer.
phrase 10
(
phrase 11
3) ... C'est une blague !
phrase 12
(
phrase 13
1) ... Yes or no ?
phrase 14
L'homme (4) :  No !
phrase 15
L'homme (5) :  Non, non je rigole pas parce qu'après c'est long !
phrase 16
(
phrase 17
Vous aimez l'aventure ?
phrase 18
La femme (6) :  L'aventure ?
phrase 19
La femme (7) :  On est tombé sur un artiste !
phrase 20
La femme (8) :
phrase 21
C'est pas là le chemin, non !
phrase 22
(
phrase 23
Je voulais vous proposer un truc c'est dommage !
phrase 24
(
phrase 25
Et vous, vous allez oser ?
phrase 26
85 000 conducteurs vivent la sensation ...
phrase 27
... 100% électrique au volant de la nouvelle NISSAN LEAF.
phrase 